# H&M Fashion Recommendation – EDA

**You can see the Key summary in the end of Notebook and All the plot are in folder figures**

Dataset bao gồm 3 bảng:
- **articles**: 105 542 sản phẩm thời trang
- **customers**: 1 371 980 khách hàng
- **transactions**: 31 788 324 giao dịch (09/2018 – 09/2020)

> **Task**: Dự đoán 12 sản phẩm mỗi khách hàng sẽ mua trong tuần cuối cùng.
> **Metric**: MAP@12 


In [ ]:
import warnings
warnings.filterwarnings("ignore")
                                  
import asyncio
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())                                  

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker
import matplotlib.dates as mdates  
import matplotlib.colors as mcolors                                
import seaborn as sns
import textwrap
from pathlib import Path

ROOT = Path.cwd().resolve().parent
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)                                                                    

sns.set_theme(style="white", palette="muted", font_scale=1.1)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
                                  
plt.rcParams["figure.dpi"] = 110

DATA = Path("../data")


In [ ]:
articles     = pd.read_parquet(DATA / "articles.parquet")
customers    = pd.read_parquet(DATA / "customers.parquet")
transactions = pd.read_parquet(DATA / "transactions.parquet")

print(f"articles     : {articles.shape}")
print(f"customers    : {customers.shape}")
print(f"transactions : {transactions.shape}")


## 1. Tổng quan Schema

In [ ]:
def df_summary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = pd.DataFrame({
        "dtype"  : df.dtypes,
        "non_null": df.notnull().sum(),
        "null_pct": (df.isnull().mean() * 100).round(2),
        "nunique": df.nunique(),
    })
    print(f"\n{'='*55}\n  {name}\n{'='*55}")
    print(summary.to_string())
    return summary

_ = df_summary(articles,     "ARTICLES")
_ = df_summary(customers,    "CUSTOMERS")
_ = df_summary(transactions, "TRANSACTIONS")


## 2. Missing Values

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (df, title) in zip(axes, [
    (articles,  "Articles"),
    (customers, "Customers"),
    (transactions, "Transactions"),
]):
    null_pct = df.isnull().mean() * 100
    null_pct = null_pct[null_pct > 0]
    if null_pct.empty:
        ax.text(0.5, 0.5, "No missing values", ha="center", va="center",
                transform=ax.transAxes, fontsize=13)
        ax.set_title(title)
    else:
        null_pct.sort_values().plot(kind="barh", ax=ax, color="salmon")
        ax.set_xlabel("Missing %")
        ax.set_title(title)
        for bar in ax.patches:
            ax.text(bar.get_width() +0.03 , bar.get_y() + bar.get_height() / 2,
                    f"{bar.get_width():.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.subplots_adjust(wspace=1)
fig.suptitle("Missing Values Overview", fontsize=18, y=1.05)
plt.savefig(FIG_DIR / "02_missing_values.png", bbox_inches="tight")
plt.show()


## 3. Transactions – Phân tích theo thời gian

In [ ]:
# 1. Tiền xử lý dữ liệu
tx = transactions.copy()
tx["year_week"] = tx["t_dat"].dt.to_period("W")

weekly = tx.groupby("year_week").agg(
    n_transactions=("article_id", "count"),
    n_customers=("customer_id", "nunique"),
    n_articles=("article_id", "nunique"),
    revenue=("price", "sum"),
).reset_index()

# Loại bỏ tuần cuối cùng nếu dữ liệu bị thiếu/sụt giảm bất thường
weekly = weekly.iloc[:-1] 

# Chuyển Period sang Datetime để Matplotlib xử lý trục thời gian tốt hơn
weekly["week_start"] = weekly["year_week"].dt.to_timestamp()

In [ ]:
# 2. Plotting
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True, gridspec_kw={'hspace': 0.15})

metrics = [
    ("n_transactions", "Transactions", "steelblue"),
    ("n_customers",    "Unique Customers", "darkorange"),
    ("n_articles",     "Unique Articles Sold", "seagreen"),
    ("revenue",        "Total Revenue (normalized)", "purple"),
]

for i, (ax, (col, label, color)) in enumerate(zip(axes, metrics)):
    # Dùng Line chart thuần túy, giảm độ đậm của fill_between
    ax.plot(weekly["week_start"], weekly[col], color=color, linewidth=2)
    ax.fill_between(weekly["week_start"], weekly[col], alpha=0.05, color=color)
    
    # Tối ưu Trục Y: Để tên trục nằm ngang phía trên biểu đồ cho thoáng
    ax.set_title(label, loc='left', fontsize=11, fontweight='bold', color='#333333', pad=5)
    
    # Thêm gridline dọc mờ
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    
    # Bỏ khung viền trên và phải cho hiện đại
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Định dạng số trục Y (thêm dấu phẩy ngăn cách hàng nghìn)
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

# 3. Tối ưu Trục X (Ngày tháng)
# Định dạng: Hiển thị Tháng - Năm (Ví dụ: Jan 2019)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
# Cách 3 tháng hiển thị 1 nhãn để tránh dày đặc
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))

plt.xticks(rotation=0, ha='center')
axes[-1].set_xlabel("", fontsize=12, labelpad=10)

plt.suptitle("WEEKLY ACTIVITY ANALYSIS (Sep 2018 – Sep 2020)", 
             fontsize=18, fontweight='bold', y=0.98)

plt.savefig(FIG_DIR / "03_weekly_activity.png", bbox_inches="tight")
plt.show()


## 4. Price sensitivity and Sale Analysis - Phân tích giá và kênh bán hàng

In [ ]:
# ── Cell Bổ sung: Price Flexibility Analysis (Single Plot) ─────────────────────

# 1. Tính toán và phân khúc dữ liệu User
# Tính hành vi giá của mỗi User (Mean & STD)
user_stats = tx.groupby("customer_id")["price"].agg(["mean", "std"]).reset_index()
user_stats.columns = ["customer_id", "user_avg_price", "user_price_std"]

# Tính tổng số lượt mua hàng của mỗi User
user_purchase_count = tx.groupby("customer_id").size().reset_index(name="total_purchases")
user_stats = user_stats.merge(user_purchase_count, on="customer_id")

# Chia nhóm khách hàng dựa trên STD (Độ lệch chuẩn của giá các món đồ họ từng mua)
# Chia làm 3 phân khúc: Thấp (mua đồ giá ổn định), Trung bình, và Cao (linh hoạt/đa dạng giá)
user_stats["std_segment"] = pd.qcut(
    user_stats["user_price_std"].fillna(0), 
    3, 
    labels=["Low (Stable)", "Medium", "High (Flexible)"]
)

# 2. Plotting (Chỉ giữ lại Boxplot phân khúc STD)
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=user_stats[user_stats["total_purchases"] < 50], # Lọc < 50 để tránh outlier làm méo biểu đồ
    x="std_segment", 
    y="total_purchases", 
    palette="viridis", 
    showfliers=False # Ẩn các điểm outlier để biểu đồ sạch sẽ, tập trung vào trung vị và quartiles
)

plt.title("Price Flexibility (STD) vs. Purchase Volume\n(High STD users often explore more products)", fontweight='bold')
plt.xlabel("User Price Standard Deviation Segment")
plt.ylabel("Total Transactions per Customer")

plt.tight_layout()
# Lưu ảnh với tên phù hợp cho báo cáo EDA của bạn
plt.savefig(FIG_DIR / "04a_user_price_flexibility.png", bbox_inches="tight")
plt.show()

# Thống kê nhanh về số lượng User ở mỗi nhóm
print("User Distribution by Price Flexibility:")
print(user_stats["std_segment"].value_counts())

In [ ]:
# ── Combined Analysis: Channel Distribution & Price Distribution ──────────────

# 1. Chuẩn bị dữ liệu
channel_counts = tx["sales_channel_id"].value_counts()
channel_labels = channel_counts.index.map({1: "Store (1)", 2: "Online (2)"})

# Lấy sample và tính toán ngưỡng để vẽ Boxplot nhanh & rõ
price_upper_limit = tx['price'].quantile(0.95)
tx_sample = tx.sample(500_000, random_state=42)

# 2. Khởi tạo Figure với 2 cột
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Subplot 1: Sales Channel Distribution (Pie Chart) ---
axes[0].pie(
    channel_counts, 
    labels=channel_labels, 
    autopct="%1.1f%%",
    colors=["#4C72B0", "#DD8452"], 
    startangle=90,
    explode=(0.05, 0) # Tạo khoảng hở nhỏ cho phần Store để tạo điểm nhấn
)
axes[0].set_title("Sales Channel Distribution", fontweight='bold')

# --- Subplot 2: Price Distribution by Channel (Boxplot) ---
# Vẽ Boxplot vào axes[1]
sns.boxplot(
    data=tx_sample, 
    x='sales_channel_id', 
    y='price', 
    palette=["#4C72B0", "#DD8452"],
    ax=axes[1],
    showfliers=False, 
    width=0.5
)

# Thêm điểm Mean (màu đỏ) để so sánh với Median
sns.pointplot(
    data=tx_sample, 
    x='sales_channel_id', 
    y='price', 
    estimator=np.mean, 
    color='red', 
    markers='D', 
    linestyles='',
    ax=axes[1]
)

# Định dạng Subplot 2
axes[1].set_xticklabels(["Store (1)", "Online (2)"])
axes[1].set_title("Price Distribution by Sales Channel", fontweight='bold')
axes[1].set_xlabel("Sales Channel")
axes[1].set_ylabel("Price")
axes[1].set_ylim(0, price_upper_limit)

# Thêm chú thích cho điểm Mean trong Boxplot

custom_lines = [Line2D([0], [0], color='red', marker='D', linestyle='None')]
axes[1].legend(custom_lines, ['Mean Price'], loc='upper right')

# 3. Hoàn thiện và lưu file
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / "04b_sales_channel_combined_analysis.png", bbox_inches="tight")
plt.show()

# 4. In thông số thống kê
print("Channel Counts:")
print(channel_counts.rename(index={1: "Store", 2: "Online"}))
print("\nDetailed Price Stats by Channel:")
print(tx.groupby('sales_channel_id')['price'].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(4))

## 5. Phân tích Khách hàng

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6)) 

# 1. Age distribution
age_valid = customers["age"].dropna()
axes[0].hist(age_valid, bins=50, color="darkorange", edgecolor="white", linewidth=0.3)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Age Distribution (n={len(age_valid):,})")
axes[0].axvline(age_valid.mean(), color="red", linestyle="--", label=f"Mean={age_valid.mean():.1f}")
axes[0].axvline(age_valid.median(), color="green", linestyle="--", label=f"Median={age_valid.median():.1f}")
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

# 2. Club membership
club = customers["club_member_status"].value_counts()
axes[1].bar(club.index, club.values, color=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_title("Club Member Status")
axes[1].set_ylabel("Count")
axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f"{bar.get_height()/1e6:.2f}M", ha="center", fontsize=10, fontweight='bold')

# 3. Fashion news frequency
fn = customers["fashion_news_frequency"].value_counts()
axes[2].bar(fn.index, fn.values, color=["#4C72B0", "#DD8452", "#55A868"])
axes[2].set_title("Fashion News Frequency")
axes[2].set_ylabel("Count")
axes[2].yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

for bar in axes[2].patches:
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f"{bar.get_height()/1e6:.2f}M", ha="center", fontsize=10, fontweight='bold')

plt.suptitle("CUSTOMER PROFILE ANALYSIS", fontsize=20, fontweight='bold', y=1.05)

plt.tight_layout()
plt.savefig(FIG_DIR / "05a_customer_profile.png", bbox_inches="tight")
plt.show()


In [ ]:
cust_purchase_count = tx.groupby("customer_id").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 6)) 

# 1. Histogram 
axes[0].hist(cust_purchase_count.clip(upper=100), bins=60,
             edgecolor="white", linewidth=0.3)
axes[0].set_xlabel("Purchases per Customer (clipped at 100)")
axes[0].set_ylabel("Count")
axes[0].set_title("Customer Purchase Frequency")

axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

# 2. CDF 
cdf = cust_purchase_count.value_counts().sort_index().cumsum() / len(cust_purchase_count)
axes[1].plot(cdf.index[:200], cdf.values[:200], linewidth=2)
axes[1].set_xlabel("Purchases per Customer")
axes[1].set_ylabel("Cumulative Fraction of Customers")
axes[1].set_title("CDF – Purchase Count")

for p in [0.5, 0.8, 0.95]:
    val = (cdf >= p).idxmax()
    axes[1].axhline(p, linestyle="--", linewidth=0.8, color="gray")
    axes[1].axvline(val, linestyle="--", linewidth=0.8, color="gray")
    axes[1].text(val + 1, p - 0.02, f"{val} purchases = {p*100:.0f}%", fontsize=9, fontweight='bold')

plt.suptitle("CUSTOMER PURCHASE BEHAVIOR", fontsize=18, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(FIG_DIR / "05b_customer_purchase_freq.png", bbox_inches="tight")
plt.show()

print("Purchase count stats per customer:")
print(cust_purchase_count.describe().round(1))


## 6. Articles Analysis - Phân tích Sản phẩm

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Product group (Top 12)
pg = articles["product_group_name"].value_counts().head(12)
axes[0, 0].barh(pg.index[::-1], pg.values[::-1], color="steelblue")
axes[0, 0].set_xlabel("Count")
axes[0, 0].set_title("Product Groups (Top 12)", fontweight='bold')
axes[0, 0].xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

# 2. Index group (department) - Pie Chart
ig = articles["index_group_name"].value_counts()
axes[0, 1].pie(ig, labels=ig.index, autopct="%1.1f%%", startangle=90,
               colors=sns.color_palette("muted", len(ig)),
               wedgeprops={'edgecolor': 'white', 'linewidth': 1})
axes[0, 1].set_title("Index Group Distribution", fontweight='bold')

# 3. Colour groups (Top 15)
cg = articles["colour_group_name"].value_counts().head(15)
axes[1, 0].barh(cg.index[::-1], cg.values[::-1], color="coral")
axes[1, 0].set_xlabel("Count")
axes[1, 0].set_title("Top 15 Colour Groups", fontweight='bold')
axes[1, 0].xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

# 4. Garment groups (Top 12)
gg = articles["garment_group_name"].value_counts().head(12)
axes[1, 1].barh(gg.index[::-1], gg.values[::-1], color="seagreen")
axes[1, 1].set_xlabel("Count")
axes[1, 1].set_title("Top 12 Garment Groups", fontweight='bold')
axes[1, 1].xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

for i, j in [(0,0), (1,0), (1,1)]:
    axes[i, j].grid(axis='x', linestyle='--', alpha=0.6)

plt.suptitle("ARTICLE & PRODUCT PROFILE ANALYSIS", fontsize=22, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(FIG_DIR / "06a_article_profile.png", bbox_inches="tight")
plt.show()


In [ ]:
article_counts = tx.groupby("article_id").size().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Article Popularity Distribution
axes[0].hist(article_counts.clip(upper=3000), bins=60,
             color="purple", edgecolor="white", linewidth=0.3)
axes[0].set_xlabel("Times Purchased (clipped at 3000)")
axes[0].set_ylabel("Count")
axes[0].set_title("Article Popularity Distribution", fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

# 2. Top 20 Most Purchased Articles
top20 = article_counts.head(20).reset_index()
top20.columns = ["article_id", "count"]
top20 = top20.merge(articles[["article_id", "prod_name"]], on="article_id", how="left")
top20["label"] = top20["prod_name"]

axes[1].barh(top20["label"][::-1], top20["count"][::-1], color="purple")
axes[1].set_xlabel("Transactions")
axes[1].set_title("Top 20 Most Purchased Articles", fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.suptitle("ARTICLE POPULARITY & SALES ANALYSIS", fontsize=18, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(FIG_DIR / "06b_article_popularity.png", bbox_inches="tight")
plt.show()

print(f"Articles with 0 purchases: {articles['article_id'].nunique() - article_counts.shape[0]:,}")
print(f"Articles with >= 1 purchase: {article_counts.shape[0]:,}")


In [ ]:
# ── Cell Bổ sung: Popularity & Trends Analysis ────────────────────────────────
# 1. Tính toán Article Trend Score (Mô phỏng logic tính Feature)
last_date = tx['t_dat'].max()
week_1 = tx[tx['t_dat'] > last_date - pd.Timedelta(days=7)]
week_4 = tx[(tx['t_dat'] > last_date - pd.Timedelta(days=28)) & (tx['t_dat'] <= last_date - pd.Timedelta(days=7))]

pop_1w = week_1.groupby('article_id').size().reset_index(name='pop_1w')
pop_4w = week_4.groupby('article_id').size().reset_index(name='pop_4w')

trend_df = pop_1w.merge(pop_4w, on='article_id', how='left').fillna(0)
trend_df['art_trend_score'] = trend_df['pop_1w'] / (trend_df['pop_4w'] / 4 + 1e-6)

# 2. Plotting
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Subplot 1: Time-series of High vs Low Trend Articles ---
high_trend_ids = trend_df[trend_df['pop_1w'] > 50].nlargest(3, 'art_trend_score')['article_id']
low_trend_ids = trend_df[trend_df['pop_1w'] > 50].nsmallest(3, 'art_trend_score')['article_id']

plot_start_date = last_date - pd.Timedelta(days=56)
recent_tx = tx[tx['t_dat'] > plot_start_date]

def plot_trend(ids, label_prefix, ax):
    for aid in ids:
        daily_sales = recent_tx[recent_tx['article_id'] == aid].groupby('t_dat').size()
        daily_sales = daily_sales.reindex(pd.date_range(plot_start_date, last_date), fill_value=0)
        ax.plot(daily_sales.index, daily_sales.rolling(7).mean(), label=f"{label_prefix} ID: {aid}", alpha=0.8)

plot_trend(high_trend_ids, "Trending Up", axes[0])
plot_trend(low_trend_ids, "Trending Down", axes[0])

axes[0].axvline(last_date - pd.Timedelta(days=7), color='red', linestyle='--', alpha=0.5, label='Last 1 Week Barrier')
axes[0].set_title("Article Sales Trend (7-day Rolling Mean)\\nHigh vs Low Trend Scores", fontweight='bold')
axes[0].set_ylabel("Daily Transactions (Smoothed)")
axes[0].legend(fontsize=9, loc='upper left')

# --- Subplot 2: Pareto Chart (Cumulative Transactions) ---
# Sắp xếp các sản phẩm theo tổng số lượng giao dịch giảm dần
article_total_tx = tx.groupby('article_id').size().sort_values(ascending=False).values
cumulative_tx = np.cumsum(article_total_tx) / np.sum(article_total_tx) * 100

axes[1].plot(cumulative_tx, color='purple', linewidth=2)
#axes[1].fill_between(range(len(cumulative_tx)), cumulative_tx, alpha=0.1, color='purple')

# Tính toán điểm 80% giao dịch
eighty_percent_idx = np.where(cumulative_tx >= 80)[0][0]
n_articles = len(article_total_tx)
perc_articles_for_80 = (eighty_percent_idx / n_articles) * 100

axes[1].axhline(80, color='orange', linestyle='--', alpha=0.7)
axes[1].axvline(eighty_percent_idx, color='orange', linestyle='--', alpha=0.7)
axes[1].text(eighty_percent_idx * 1.1, 40, f"Top {perc_articles_for_80:.1f}% articles\\naccount for 80% of total transactions", 
             fontweight='bold', color='darkorange')

axes[1].set_title("Transaction Concentration (Pareto Curve)\\nCumulative % of Total Transactions", fontweight='bold')
axes[1].set_xlabel("Number of Articles (Ranked by popularity)")
axes[1].set_ylabel("Cumulative % of Total Transactions")
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig(FIG_DIR / "06c_popularity_trend_analysis.png", bbox_inches="tight")
plt.show()

## 7. Phân tích chéo: Độ tuổi × Nhóm sản phẩm

In [ ]:
tx = transactions.copy()
tx["year_month"] = tx["t_dat"].dt.to_period("M")
bins = [15, 25, 35, 45, 55, 65, 100]
labels = ["16-24", "25-34", "35-44", "45-54", "55-64", "65+"]
customers["age_group"] = pd.cut(customers["age"], bins=bins, labels=labels, right=True)

tx_cust = tx.merge(customers[["customer_id", "age_group"]], on="customer_id", how="left")
tx_art  = tx_cust.merge(articles[["article_id", "product_group_name"]], on="article_id", how="left")

pivot = (
    tx_art.dropna(subset=["age_group"])
    .groupby(["age_group", "product_group_name"])
    .size()
    .unstack(fill_value=0)
)
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0) * 100

top_groups = articles["product_group_name"].value_counts().head(8).index
pivot_plot = pivot_norm[top_groups]

wrapped_labels = [textwrap.fill(label, width=12) for label in pivot_plot.columns]

fig, ax = plt.subplots(figsize=(15, 8))

sns.heatmap(
    pivot_plot, 
    annot=True, 
    fmt=".1f", 
    cmap="YlOrRd", 
    ax=ax,
    linewidths=0.5, 
    cbar_kws={"label": "Row % of purchases"},
    annot_kws={"size": 12} 
)

ax.set_xticklabels(wrapped_labels, rotation=0, ha='center')

ax.set_title("Purchase % by Age Group × Product Group (top 8 groups)", pad=20, fontsize=14)
ax.set_xlabel("Product Group", fontsize=12)
ax.set_ylabel("Age Group", fontsize=12)

plt.tight_layout()
plt.savefig(FIG_DIR / "07_age_product_heatmap.png", bbox_inches="tight")
plt.show()


## 8. Repurchase Behaviors - Phân tích hành vi mua hàng

In [ ]:
# For each customer, what fraction of purchases are repeat items?
cust_art = tx.groupby(["customer_id", "article_id"]).size().reset_index(name="n")
repeat_rate = (cust_art.groupby("customer_id")
               .apply(lambda g: (g["n"] > 1).mean(), include_groups=False)
               .reset_index(name="repeat_rate"))

fig, ax = plt.subplots(figsize=(10, 5))

# Histogram
ax.hist(repeat_rate["repeat_rate"], bins=50, color="teal", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Repeat Purchase Fraction (0 to 1)")
ax.set_ylabel("Number of Customers")
ax.set_title(f"Customer Loyalty: Repeat Purchase Rate Analysis (Mean: {repeat_rate['repeat_rate'].mean():.2%})", 
             fontweight='bold', fontsize=12)

ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

plt.tight_layout()
plt.savefig(FIG_DIR / "08a_repeat_purchase.png", bbox_inches="tight")
plt.show()

print(f"Customers who never repeat: {(repeat_rate['repeat_rate'] == 0).mean():.1%}")
print(f"Customers with >50% repeat: {(repeat_rate['repeat_rate'] > 0.5).mean():.1%}")


In [ ]:
# 1. Prepare data for subplot 1: Rentention rate monthly
# ── Monthly Customer Return Rate (customer-level) ────────────────────
tx = transactions.copy()
tx["year_month"] = tx["t_dat"].dt.to_period("M")

# khách active mỗi tháng
monthly_users = tx.groupby("year_month")["customer_id"].unique()

return_rates = []

for month, users in monthly_users.items():
    users = set(users)
    
    # khách đã từng mua trước tháng này
    prev_users = set(
        tx[tx["year_month"] < month]["customer_id"]
    )
    
    returning_users = users & prev_users
    
    rate = len(returning_users) / len(users)
    
    return_rates.append({
        "year_month": month,
        "return_rate": rate
    })

monthly_ret = pd.DataFrame(return_rates)
monthly_ret["year_month"] = monthly_ret["year_month"].dt.to_timestamp()

In [ ]:
# 2. Prepare data for subplot 2: repurchase components
# ── Repurchase Pattern by Attribute ─────────────────────────────────────

# merge article features
tx_rep = tx.merge(
    articles[[
        "article_id",
        "product_code",
        "product_type_no",
        "colour_group_code",
        "product_group_name"
    ]],
    on="article_id",
    how="left"
)

# sort theo thời gian để xác định lần mua tiếp theo
tx_rep = tx_rep.sort_values(["customer_id", "t_dat"])

# shift để lấy lần mua tiếp theo của cùng customer
tx_rep["next_article"] = tx_rep.groupby("customer_id")["article_id"].shift(-1)
tx_rep["next_product_code"] = tx_rep.groupby("customer_id")["product_code"].shift(-1)
tx_rep["next_product_type"] = tx_rep.groupby("customer_id")["product_type_no"].shift(-1)
tx_rep["next_colour"] = tx_rep.groupby("customer_id")["colour_group_code"].shift(-1)
tx_rep["next_group"] = tx_rep.groupby("customer_id")["product_group_name"].shift(-1)

# chỉ giữ các dòng có next purchase
df_pairs = tx_rep.dropna(subset=["next_article"]).copy()

# ── Define repurchase patterns
df_pairs["same_article"] = (df_pairs["article_id"] == df_pairs["next_article"]).astype(int)
df_pairs["same_product_code"] = (df_pairs["product_code"] == df_pairs["next_product_code"]).astype(int)
df_pairs["same_product_type"] = (df_pairs["product_type_no"] == df_pairs["next_product_type"]).astype(int)
df_pairs["same_colour"] = (df_pairs["colour_group_code"] == df_pairs["next_colour"]).astype(int)
df_pairs["same_group"] = (df_pairs["product_group_name"] == df_pairs["next_group"]).astype(int)

# ── Compute ratios
rep_stats = pd.Series({
    "Same Article": df_pairs["same_article"].mean(),
    "Same Product Code": df_pairs["same_product_code"].mean(),
    "Same Product Type": df_pairs["same_product_type"].mean(),
    "Same Colour": df_pairs["same_colour"].mean(),
    "Same Product Group": df_pairs["same_group"].mean(),
}).sort_values(ascending=False)


In [ ]:
# 3. Vẽ 
# ── 1. Khởi tạo Figure với 2 cột (1 hàng, 2 cột)
# facecolor='none' để hỗ trợ nền trong suốt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7),facecolor='white')

# ── 2. PLOT BÊN TRÁI: Monthly Return Rate (Line Chart)
monthly_ret = monthly_ret.sort_values("year_month")
ax1.plot(
    monthly_ret["year_month"], 
    monthly_ret["return_rate"], 
    marker="o", linewidth=2, color="#0994B3" # Màu đỏ đậm cho đồng bộ
)

ax1.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax1.set_title("Monthly Customer Return Rate", fontweight='bold', fontsize=14)
ax1.set_xlabel("Month")
ax1.set_ylabel("Return Rate")
ax1.set_facecolor('none')
ax1.grid(axis='y', alpha=0.2) # Chỉ để grid ngang cho thoáng


# ── 3. PLOT BÊN PHẢI: Repurchase Behavior (Bar Chart - Gradient Red)
# Thiết lập dải màu đỏ đen đậm -> nhạt
colors = [ "#99F0E4","#0BBCB1","#03A19C"] 
cmap = mcolors.LinearSegmentedColormap.from_list("custom_red", colors)

plot_data = rep_stats.sort_values() # Nhỏ trước lớn sau
bar_colors = cmap(np.linspace(0.3, 1, len(plot_data))) # Tạo gradient

ax2.barh(plot_data.index, plot_data.values, color=bar_colors)
ax2.set_title("Repurchase Behavior Across Product Attributes", fontweight='bold', fontsize=14)
ax2.set_xlabel("Fraction of consecutive purchases")
ax2.set_xlim(0, plot_data.max() * 1.2)
ax2.set_facecolor('none')

# Annotate % cho bar chart
for i, v in enumerate(plot_data.values):
    ax2.text(v + 0.005, i, f"{v:.1%}", va="center", fontsize=11, fontweight='bold', color="#0B7CA9")


# ── 4. TINH CHỈNH CHUNG
sns.despine(fig=fig) # Xóa khung trên và phải cho cả 2 axes

# Làm sạch các đường viền còn lại của trục tọa độ để tăng độ "trong suốt"
for ax in [ax1, ax2]:
    ax.tick_params(axis='both', which='both', length=0) # Bỏ dấu gạch nhỏ trên trục

plt.tight_layout()

# Lưu file với nền trong suốt
plt.savefig(FIG_DIR / "08b_combined_analysis.png", transparent=False, dpi=110)
plt.show()

## 9. Recency Analysis - Phân tích hành vi mua hàng theo recency days

In [ ]:
# ── Recency vs Purchase Behavior (Repurchase vs Popularity) ─────────────

# 1. Chọn snapshot date (cuối train)
snapshot_date = tx["t_dat"].max() - pd.Timedelta(days=7)

# 2. Lấy history trước snapshot
hist = tx[tx["t_dat"] < snapshot_date].copy()

# 3. Tính recency cho mỗi user tại snapshot
last_purchase = (
    hist.groupby("customer_id")["t_dat"]
    .max()
    .reset_index(name="last_purchase_date")
)

last_purchase["recency_days"] = (
    snapshot_date - last_purchase["last_purchase_date"]
).dt.days

# 4. Binning recency
def recency_bucket(x):
    if x <= 5:
        return "1–5 days"
    elif x <= 10:
        return "5–10 days"
    else:
        return ">10 days"

last_purchase["recency_group"] = last_purchase["recency_days"].apply(recency_bucket)

# 5. Lấy transactions sau snapshot (proxy "future behavior")
future = tx[tx["t_dat"] >= snapshot_date].copy()

# 6. Merge recency
future = future.merge(
    last_purchase[["customer_id", "recency_group"]],
    on="customer_id",
    how="left"
)

# 7. Tạo user history set (để check repurchase)
user_hist = (
    hist.groupby("customer_id")["article_id"]
    .apply(set)
    .to_dict()
)

future["is_repurchase"] = future.apply(
    lambda row: row["article_id"] in user_hist.get(row["customer_id"], set()),
    axis=1
)

# 8. Popular articles (top 1%)
top_articles = (
    hist["article_id"]
    .value_counts()
    .head(int(0.01 * hist["article_id"].nunique()))
    .index
)

future["is_popular"] = future["article_id"].isin(top_articles)

# 9. Aggregate theo recency group
recency_stats = (
    future.groupby("recency_group")
    .agg(
        repurchase_rate=("is_repurchase", "mean"),
        popular_rate=("is_popular", "mean"),
        n=("article_id", "count")
    )
    .reset_index()
)

# 10. Plot
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(recency_stats))
width = 0.35

ax.bar(x - width/2, recency_stats["repurchase_rate"], width, label="Repurchase", color="teal")
ax.bar(x + width/2, recency_stats["popular_rate"], width, label="Popularity", color="orange")

ax.set_xticks(x)
ax.set_xticklabels(recency_stats["recency_group"])
ax.set_ylabel("Fraction of purchases")
# ax.set_title(
#     "Recency and Purchase Behavior",
#     fontweight="bold"
# )

# annotate %
for i in range(len(recency_stats)):
    ax.text(x[i] - width/2, recency_stats["repurchase_rate"].iloc[i] + 0.005,
            f"{recency_stats['repurchase_rate'].iloc[i]:.1%}", ha="center", fontsize=9)
    ax.text(x[i] + width/2, recency_stats["popular_rate"].iloc[i] + 0.005,
            f"{recency_stats['popular_rate'].iloc[i]:.1%}", ha="center", fontsize=9)

# Thay dòng ax.legend() bằng:
ax.legend(title="Type", loc='upper left', bbox_to_anchor=(1, 1))
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / "09_recency_behavior.png", bbox_inches="tight")
plt.show()

print(recency_stats)


## 10. Phân tích Mùa vụ (Seasonality)

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
                                        
ROOT = Path.cwd().resolve().parent
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)                                                                    

sns.set_theme(style="white", palette="muted", font_scale=1.1)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
                                  
plt.rcParams["figure.dpi"] = 110

DATA = Path("../data")


articles     = pd.read_parquet(DATA / "articles.parquet")
transactions = pd.read_parquet(DATA / "transactions.parquet")


In [ ]:
# PCA + GMM clustering on monthly sales patterns
tx_temp = transactions.copy()
tx_temp["month_start"] = tx_temp["t_dat"].dt.to_period("M").dt.to_timestamp()

tx_temp["month_year"] = tx_temp["t_dat"].dt.to_period("M")
df_cat = tx_temp.merge(
    articles[["article_id", "index_group_name", "index_name", "product_type_name"]],
    on="article_id"
)

df_cat["Category"] = (
    df_cat["index_group_name"] + " | " +
    df_cat["index_name"] + " | " +
    df_cat["product_type_name"]
)

cat_totals = df_cat.groupby("Category").size()
monthly_cat = df_cat.groupby(["Category", "month_year"]).size().reset_index(name="cnt")

monthly_cat["pct"] = monthly_cat.apply(
    lambda r: 100 * r["cnt"] / cat_totals[r["Category"]], axis=1
)
monthly_cat["my_str"] = monthly_cat["month_year"].astype(str)
pca_data = monthly_cat.pivot(index="Category", columns="my_str", values="pct").fillna(0)

scaled = StandardScaler().fit_transform(pca_data)
pcs = PCA(n_components=2, random_state=42).fit_transform(scaled)

gmm = GaussianMixture(n_components=4, random_state=42)
labels = gmm.fit_predict(pcs)
pca_df = pd.DataFrame({"PC1": pcs[:, 0], "PC2": pcs[:, 1], "Season": labels})
season_map = {0: "No season", 1: "Spring/Summer", 2: "Autumn/Winter", 3: "Mixed"}

fig, ax = plt.subplots(figsize=(10, 6)) 

for s, name in season_map.items():
    sub = pca_df[pca_df["Season"] == s]
    ax.scatter(sub["PC1"], sub["PC2"], label=name, s=25, alpha=0.7, edgecolors='w')

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA of Category Sales Trends – GMM Clusters (Seasonality)", pad=15)

ax.legend(
    title="Seasonality Type", 
    loc='upper left', 
    bbox_to_anchor=(1.02, 1), 
    borderaxespad=0,
    frameon=True
)

sns.despine()

plt.tight_layout()
plt.savefig(FIG_DIR / "10_pca_seasonality.png", bbox_inches="tight", dpi=300)
plt.show()

season_counts = pca_df["Season"].value_counts().rename(index=season_map)
print("-" * 30)
print("Thống kê số lượng Category theo mùa:")
print(season_counts)
print("-" * 30)


## 11. Sản phẩm Out-of-Stock (Discontinued)

In [ ]:
tx_temp2 = transactions.copy()
tx_temp2["before_2019"] = tx_temp2["t_dat"] < "2019-01-01"

total_sales = tx_temp2.groupby("article_id").size()
old_sales   = tx_temp2[tx_temp2["before_2019"]].groupby("article_id").size()
ratio = (old_sales / total_sales).fillna(0).rename("before_2019_ratio")

article_sales = pd.DataFrame({"total_sales": total_sales, "before_2019_ratio": ratio})
obsolete = article_sales[article_sales["before_2019_ratio"] >= 0.95]

print(f"Tổng articles trong transactions : {len(article_sales):,}")
print(f"Obsolete (≥95% doanh số trước 2019): {len(obsolete):,}  ({len(obsolete)/len(article_sales):.1%})")

# 1. gridspec_kw={'width_ratios': [1, 1.5]} 
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.2]})

# --- SUBPLOT 1: HISTOGRAM ---
axes[0].hist(article_sales["before_2019_ratio"], bins=50, color="steelblue",
             edgecolor="white", linewidth=0.3)
axes[0].axvline(0.95, color="red", linestyle="--", label="Threshold 0.95")
axes[0].get_yaxis().set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ',')))

axes[0].set_xlabel("Fraction of sales before 2019")
axes[0].set_ylabel("Number of Articles")
axes[0].set_title("Distribution of Old-Sales Ratio")

# legend
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1)

# --- SUBPLOT 2: PIE CHART ---
sizes = [len(obsolete), len(article_sales) - len(obsolete)]
axes[1].pie(sizes, labels=["Discontinued", "Active"],
            autopct="%1.1f%%", colors=["salmon", "steelblue"], 
            startangle=90, pctdistance=0.85, explode=(0.05, 0))
axes[1].set_title("Active vs Discontinued Articles")

plt.tight_layout()
plt.savefig(FIG_DIR / "11_out_of_stock.png", bbox_inches="tight")
plt.show()


## 12. Repurchase Rate theo Granularity

In [ ]:
df_rep = transactions.merge(
    articles[["article_id", "product_code", "product_group_name"]],
    on="article_id", how="left"
)

def repurchase_pct(df: pd.DataFrame, item_col: str, weeks: list[int] = [1, 2, 3]) -> list[float]:
    df = df.sort_values(["customer_id", item_col, "t_dat"]).copy()
    df["next_purchase"] = df.groupby(["customer_id", item_col])["t_dat"].shift(-1)
    df["days_to_next"]  = (df["next_purchase"] - df["t_dat"]).dt.days
    total = len(df)
    return [
        100 * ((df["days_to_next"] > 0) & (df["days_to_next"] <= 7 * w)).sum() / total
        for w in weeks
    ]

weeks = [1, 2, 3]
article_rep  = repurchase_pct(df_rep, "article_id",        weeks)
product_rep  = repurchase_pct(df_rep, "product_code",      weeks)
category_rep = repurchase_pct(df_rep, "product_group_name", weeks)

plot_df = pd.DataFrame(
    {"Article ID": article_rep, "Product Code": product_rep, "Category": category_rep},
    index=[f"Within week {w}" for w in weeks],
)

x = np.arange(len(plot_df.index))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
for i, col in enumerate(plot_df.columns):
    ax.bar(x + (i - 1) * width, plot_df[col], width, label=col)
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index)
ax.set_ylabel("Repurchase Rate (%)")
ax.set_title("Repurchase Rate by Granularity & Time Window")
ax.legend(title="Granularity")
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / "12_repurchase_granularity.png", bbox_inches="tight")
plt.show()

print(plot_df.round(2))


## 13. Word Cloud – Mô tả Sản phẩm

In [ ]:
try:
    from wordcloud import WordCloud, STOPWORDS
    import re

    def clean_text(text: str) -> str:
        text = text.lower()
        text = re.sub(r"[^a-z\s]", " ", text)
        text = re.sub(r"\s+", " ", text)
        return text

    corpus = " ".join(
        articles["detail_desc"].dropna().astype(str).apply(clean_text)
    )
    stop_words = set(STOPWORDS) | {"cm", "product", "item", "size", "wear", "made"}

    wc = WordCloud(
        width=1200, height=500, background_color="white",
        stopwords=stop_words, max_words=200, collocations=False,
    ).generate(corpus)

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title("Word Cloud – Article Detail Descriptions", fontsize=14)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "13_wordcloud.png", bbox_inches="tight")
    plt.show()
except ImportError:
    print("wordcloud not installed — run: uv add wordcloud")


## **Tóm tắt Insights từ Exploratory Data Analysis**

| # | Insight (Dữ liệu & Phát hiện) | Ý nghĩa cho Recommendation System |
|---|-------------------------------|-----------------------------------|
| 1 | **Data scale & sparsity**: 31.8M transactions, 1.37M customers, 105.5K articles; median user chỉ có **9 purchases/2 năm**, 80% users < 34 purchases | CF truyền thống dễ bị sparse → cần kết hợp popularity baseline + candidate generation đa nguồn |
| 2 | **Repurchase behavior mạnh**: 60-80% active users là returning customers; ~50% consecutive purchases cùng product group, ~30% cùng product type/color | **Repurchase là baseline mạnh nhất** (MAP@12=0.0241); cần 3 nguồn candidate: repurchase-short, repurchase-long, same product-code variant |
| 3 | **Catalogue turnover nhanh**: ~10.3% articles discontinued (≥95% sales trước 2019); top 20.7% articles chiếm 80% transactions (Pareto) | Áp dụng **out-of-stock mask** để filter discontinued items; ưu tiên **short-window popularity** (14d) thay vì all-time |
| 4 | **Fashion preferences non-stationary**: Recent popularity (2w) MAP@12=0.0068 vs Global popularity 0.0029 (**2.3x improvement**) | Feature `art_pop_1w`, `art_pop_2w`, `art_trend_score` quan trọng hơn `art_pop_all`; cập nhật feature window hàng ngày |
| 5 | **Recency-driven intent**: Users với recency 1-5 days ưu tiên repurchase; users >10 days inactive chuyển sang popular items | Feature `user_days_since_last_tx`, `ua_days_since_purchase` giúp model adapt strategy theo mức độ active của user |
| 6 | **Price sensitivity phân hóa**: Price-flexible users (high STD) có volume mua cao hơn; online channel (70.4%) có mean price cao, sensitivity thấp hơn offline | Dùng `user_price_std`, `user_avg_price`, `ua_price_affinity`, `user_online_ratio` để cá nhân hóa theo spending pattern và channel |
| 7 | **Age segmentation có ý nghĩa**: 16-24 tuổi mua đa dạng (38% upper-body + swimwear/underwear); 45+ tập trung vào basic apparel (~49% upper-body), giảm discretionary categories | Chia 6 age bins (16-24, 25-34, ..., 65+) làm feature + key cho age-segmented popularity candidate source |
| 8 | **Cold-start phổ biến**: ~60% test users có AP@12=0, chủ yếu là users với ≤2 prior purchases | Cần fallback strategy: age-segmented popularity + content-based features (future work: image/text embeddings) |
| 9 | **Seasonality implicit**: PCA+GMM phát hiện 4 seasonal patterns align với spring/summer/autumn/winter cycles | Không model seasonality trực tiếp, nhưng exploit qua short-window popularity & trend features (`art_trend_score`) |
| 10| **Feature importance balance**: `art_avg_price` quan trọng nhất, tiếp theo là price-affinity features, short-term popularity, recency signals | Ranker học được sự tương tác giữa price positioning, temporal dynamics và user behavior → không chỉ dựa vào interaction-only signals |

---

### 💡 **Key Takeaways cho Modeling**

1. ✅ **Repurchase signal** là "low-hanging fruit" → luôn include làm candidate source  
2. ✅ **Short-term > Long-term popularity** trong fashion domain  
3. ✅ **Recency & price features** giúp adapt recommendation theo context user  
4. ✅ **Age segmentation** có giá trị khi kết hợp với behavioral signals  
5. ⚠️ **Cold-start users** cần special handling (fallback + content-based future work)  
6. ⚠️ **Out-of-stock mask** là bắt buộc để avoid recommend discontinued items